## xgb & lgb & randomforest ensemble

## Feature Select & Engineering

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree

#데이터셋 불러오기
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 고윳값 Drop
X_train=train_df.drop(["index","name","health_status"],axis=1)
X_test=test_df.drop(["index","name"],axis=1)
y_train=train_df['health_status']

# 주요 피처 리스트
important_features = ['iron_lvl', 'stature_inch', 'fat_content', 'age_enzyme_ratio', 'Total_Cholesterol']

def create_interaction_features(df):
    df['bmi'] = df['mass_kg'] / ((df['stature_inch'] + 100) / 100) ** 2
    df['waist_to_age_ratio'] = df['midsection_inch'] / 2024-df['birth']
    df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)
    df['bmi_iron'] = df['bmi'] * df['iron_lvl']
    df['iron_stature'] = df['iron_lvl'] * df['stature_inch']
    df['mass_iron_ratio'] = df['mass_kg'] / df['iron_lvl']
    df['fat_chol_ratio'] = df['fat_content'] / (df['chol_good'] + df['chol_bad'])
    df['chol_good_iron_mult'] = df['chol_good'] * df['iron_lvl']
    df['lipid_enzyme3_mult'] = df['lipid_lvl'] * df['enzyme_3']
    
    return df
# X_train, X_test 데이터에 적용
X_train = create_interaction_features(X_train)
X_test = create_interaction_features(X_test)

# 결과 확인
X_test


In [ ]:
# X_train['audio_r']=X_train['audio_r'].astype('category')
# X_train['audio_l']=X_train['audio_l'].astype('category')

# X_test['audio_l']=X_test['audio_l'].astype('category')
# X_test['audio_r']=X_test['audio_r'].astype('category')

drop=['audio_l','audio_r']
X_train=X_train.drop(drop,axis=1)
X_test=X_test.drop(drop,axis=1)

In [ ]:
X_train.info()

## hyperparameter Optimizer

### lgb boost randomsearch

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import os
import lightgbm as lgb

model = lgb.LGBMClassifier(random_state=10)

params = {
    'learning_rate' : [0.05, 0.07,0.08,0.09,0.1],  # 학습률
    'n_estimators': [000,1200,1400,1500],      # 트리 개수

    'max_depth': [3,4,5,6],                  # 최대 트리 깊이
    'num_leaves': [40, 43,44,45,46,47,50],             # 리프 노드 개수
    'min_data_in_leaf': [50,51,52,53,54,55,56,58,60],      # 최소 리프 데이터 개수
    'min_sum_hessian_in_leaf': [0.01,0.1, 0.8,1,1.2,5],  # 리프 내 최소 헷시안 합

    'lambda_l1': [10, 11,12,13,14,15],            # L1 정규화
    'lambda_l2': [15,16,17,18,19,20, 21,22,23,25],            # L2 정규화
    
    'subsample': [0.8,0.9,0.95, 1.0],              # 데이터 샘플링 비율
    'subsample_freq': [3,4,5,6,7,8],                # 샘플링 빈도
    'colsample_bytree': [0.8, 0.9, 1.0],       # 피처 샘플링 비율
    'feature_fraction': [0.5,0.55,0.6],  # 피처 비율
    'bagging_fraction': [0.6, 0.7, 0.8, 0.9, 1.0],  # 배깅 비율
    
    'min_gain_to_split': [1.0],      # 분할 시 최소 이득
    'max_bin': [450,470,500,530,550],                     # 히스토그램 bin 개수
    
    'boosting_type': ['gbdt', 'dart', 'goss'],      # 부스팅 방식
    'random_state': [100,20,10]                   # 시드 값 (결과 재현성)
}

random_search_lgb = RandomizedSearchCV(model, params, n_iter=30,scoring='roc_auc', verbose=3,cv=3)
random_search_lgb.fit(X_train, y_train)

# Extract results from the GridSearchCV output
cv_results = random_search_lgb.cv_results_

# Write the extracted results to the file

# Define the filename to save the results
filename = 'lgboost.txt'
# Write the extracted results to the file (create file if it doesn't exist)
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")
        
    f.write("\nBest Parameters:\n")
    f.write(f"{random_search_lgb.best_params_}\n")
    f.write(f"Best Score: {random_search_lgb.best_score_:.3f}\n")
    f.write(f"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")


### xgb boost randomsearch

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import os
import xgboost as xgb


model = xgb.XGBClassifier(random_state=10)

# Define LightGBM parameters for GridSearchCV
params = {
    'learning_rate': [0.05,0.09,0.1, 0.12, 0.15, 0.18, 0.2],
    'max_depth': [3,4,5, 6, 7],
    'n_estimators': [600,650,700,750,800],
    'subsample': [0.85, 0.9, 0.95,1.0],
}

random_search_xgb = RandomizedSearchCV(model, params, n_iter=10,scoring='roc_auc', verbose=3,cv=3)
random_search_xgb.fit(X_train, y_train)

# Extract results from the GridSearchCV output
cv_results = random_search_xgb.cv_results_

# Write the extracted results to the file

# Define the filename to save the results
filename = 'xgboost.txt'
# Write the extracted results to the file (create file if it doesn't exist)
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")
        
    f.write("\nBest Parameters:\n")
    f.write(f"{random_search_xgb.best_params_}\n")
    f.write(f"Best Score: {random_search_xgb.best_score_:.3f}\n")
    f.write(f"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")



### randomforest randomsearch

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)
param_grid={
    'n_estimators':[400,600,800,1000],
    'max_depth':[8,10,12,14,16,18,20],
    'class_weight': [None],
    'max_features': ['sqrt','log2'],
    'min_samples_split': [1,2,3,4,5,6]
    
}

randomSearch=RandomizedSearchCV(model, param_grid,scoring='roc_auc',cv=2,verbose=3, n_jobs=4)

randomSearch.fit(X_train,y_train)

best_model = randomSearch.best_estimator_
pred=best_model.predict(X_test)

In [ ]:
print(randomSearch.best_params_)

##  model 정의 및 학습

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# XGBoost 모델 정의
xgb_model = XGBClassifier(
    subsample=0.85,
    n_estimators=750,
    max_depth=5,
    learning_rate=0.05,
)

# LightGBM 모델 정의
lgb_model = LGBMClassifier(
    subsample_freq=5,
    subsample=1.0,
    random_state=100,
    num_leaves=45,
    n_estimators=1250,
    min_sum_hessian_in_leaf=1,
    min_gain_to_split=1.0,
    min_data_in_leaf=55,
    max_depth=4,
    max_bin=500,
    learning_rate=0.07,
    lambda_l2=20,
    lambda_l1=13,
    feature_fraction=0.6,
    colsample_bytree=0.9,
    boosting_type='gbdt',
    bagging_fraction=0.7
)

# Random Forest 모델 정의
rf_model = RandomForestClassifier(
    n_estimators=800,
    min_samples_split=6,
    max_features='sqrt',
    max_depth=14,
    class_weight=None,
    random_state=42
)

# 모델 확인
print("XGBClassifier:", xgb_model)
print("LGBMClassifier:", lgb_model)
print("RandomForestClassifier:", rf_model)


In [ ]:
xgb_model.fit(X_train,y_train)
lgb_model.fit(X_train,y_train)
rf_model.fit(X_train,y_train)

## ensemble

In [ ]:
lgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_probs = lgb_model.predict_proba(X_test)[:, 1]
rf_probs=rf_model.predict_proba(X_test)[:, 1]
# 예측 확률 값 출력
print('best [0.53321308 0.02115297 0.03852971 ... 0.06604431 0.12076061 0.83912494]')
print('lgb ',lgb_probs)
print('xgb ',xgb_probs)
print('rf  ',rf_probs)

#### 0.5 기준 앙상블

In [ ]:
# import numpy as np  # numpy가 import되어 있는지 확인


# def ensemble_predict(xgb_probs, lgb_probs):
#     # 두 모델의 확률을 기준으로 결합
#     ensemble_probs = []
    
#     for xgb_prob, lgb_prob in zip(xgb_probs, lgb_probs,rf_probs):
#         if xgb_prob < 0.5 and lgb_prob < 0.5 and rf_probs:
#             ensemble_probs.append(min(xgb_prob, lgb_prob))
#         elif xgb_prob >= 0.5 and lgb_prob >= 0.5:
#             ensemble_probs.append(max(xgb_prob, lgb_prob))
#         else:
#             ensemble_probs.append(max(xgb_prob, lgb_prob))  # 하나만 0.5 이상일 경우
            
#     return np.array(ensemble_probs)

# # 앙상블 예측 수행
# ensemble_probs = ensemble_predict(xgb_probs, lgb_probs)

# # 앙상블 확률 출력
# print('best [0.53321308 0.02115297 0.03852971 ... 0.06604431 0.12076061 0.83912494]')
# print("Ensemble Probabilities:", ensemble_probs)

import numpy as np

def ensemble_probs(xgb_probs, lgb_probs, rf_probs):
    ensemble_probs = []
    for xgb_prob, lgb_prob, rf_prob in zip(xgb_probs, lgb_probs, rf_probs):
        # 세 개 다 0.5를 넘는 경우
        if xgb_prob > 0.5 and lgb_prob > 0.5 and rf_prob > 0.5:
            ensemble_probs.append(max(xgb_prob, lgb_prob, rf_prob))
        # 세 개 중 두 개가 0.5를 넘는 경우
        elif (xgb_prob > 0.5 and lgb_prob > 0.5) or (xgb_prob > 0.5 and rf_prob > 0.5) or (lgb_prob > 0.5 and rf_prob > 0.5):
            over_0_5 = [prob for prob in (xgb_prob, lgb_prob, rf_prob) if prob > 0.5]
            ensemble_probs.append(max(over_0_5))
        # 세 개 중 두 개가 0.5보다 아래인 경우
        elif (xgb_prob <= 0.5 and lgb_prob <= 0.5) or (xgb_prob <= 0.5 and rf_prob <= 0.5) or (lgb_prob <= 0.5 and rf_prob <= 0.5):
            under_0_5 = [prob for prob in (xgb_prob, lgb_prob, rf_prob) if prob <= 0.5]
            ensemble_probs.append(min(under_0_5))
        # 세 개 다 0.5를 넘지 못하는 경우
        else:
            ensemble_probs.append(min(xgb_prob, lgb_prob, rf_prob))
    
    return np.array(ensemble_probs)

ensemble_probs_1 = ensemble_probs(xgb_probs, lgb_probs, rf_probs)
print(ensemble_probs_1)

# 1. 0.85048

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = ensemble_probs_1
submission_df.to_csv('submission.csv', index=False)

#### 가중치 기반 앙상블

In [ ]:
def weighted_average_predict(xgb_probs, lgb_probs, rf_probs, weights=(0.4, 0.4, 0.2)):
    # 가중치 기반 확률 평균 계산
    return weights[0] * xgb_probs + weights[1] * lgb_probs + weights[2] * rf_probs

# 예시: XGBoost, LightGBM, RandomForest 모델의 확률
ensemble_probs_2 = weighted_average_predict(xgb_probs, lgb_probs, rf_probs, weights=(0.4, 0.3, 0.3))
print('XGB: 30%, LGB: 60%, RF: 10%', ensemble_probs_2)


In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = ensemble_probs_2
submission_df.to_csv('submission.csv', index=False)

#### 스태킹

In [ ]:
from sklearn.ensemble import StackingClassifier
estimators = [
    ('xgb', xgb_model),
    ('lgb', lgb_model),
    #('rf', rf_model)
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),  # 메타 모델
    cv=5,  # 내부 교차 검증
    stack_method='predict_proba'  # 기본 모델의 확률값 사용
)

stacking_model.fit(X_train, y_train)

In [ ]:
stacking_model_probs = stacking_model.predict_proba(X_test)[:, 1]
print(stacking_model_probs )

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = stacking_model_probs
submission_df.to_csv('submission.csv', index=False)

#### 평균 앙상블

In [ ]:
# 평균 앙상블
ensemble_probs = (xgb_probs + lgb_probs) / 3

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = ensemble_probs
submission_df.to_csv('submission.csv', index=False)

In [ ]:
lgb_model = LGBMClassifier(
    subsample=0.95,
    num_leaves=50,
    n_estimators=750,
    max_depth=5,
    learning_rate=0.12,
    lambda_l2=30,
    lambda_l1=15,
)
lgb_model.fit(X_train,y_train)

In [ ]:
lgb_probs = xgb_model.predict_proba(X_test)[:, 1]

### 성능 예측,,?

In [ ]:
import numpy as np

# 앙상블 확률을 기반으로 성능 평가
def evaluate_ensemble(ensemble_probs_1, ensemble_probs_2):
    # 두 앙상블 확률 차이 계산
    auc_difference = np.abs(ensemble_probs_1 - ensemble_probs_2).mean()
    
    # 출력
    print("Average Probability Difference:", auc_difference)



# 성능 평가 수행
evaluate_ensemble(ensemble_probs_1, lgb_probs)


- Average Probability Difference: 0.03146682687004264 (0.85048 vs 가중치기반(0.4,0.3,0.3))
- Average Probability Difference: 0.040113029117404754(0.85048 vs 스태킹(xgb,lgb))